# Adversarial Phishing Detection
## Real-Time XAI API with Active Hardening
**Full Pipeline Notebook**

In [ ]:
!pip install -q transformers datasets scikit-learn xgboost shap lime fastapi uvicorn pyngrok joblib textstat nltk sentence-transformers

## Phase 1 & 2: Dataset, Classical & Transformer Models

In [ ]:
import pandas as pd, numpy as np, re, os, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Add local import for consistency with API
import sys
sys.path.append('..')
from api.utils import extract_features

# 1. Load Datasets
adv_path = '../data/adversarial_samples.csv'
adv_df = pd.read_csv(adv_path)

# Legitimate samples (expanded for balance)
legit_emails = [
    "Hi Team, just a reminder for our sync.", "The report is attached.",
    "Updated the roadmap on the drive.", "Are we still on for lunch?",
    "Great job on the presentation!", "Meeting notes from this morning."
] * 84 # ~500 samples

legit_df = pd.DataFrame({'text': legit_emails, 'label': 0})
df = pd.concat([adv_df, legit_df]).reset_index(drop=True)

# 2. Feature Extraction
print("Extracting features...")
ling_features_list = [extract_features(t) for t in df['text']]
ling_df = pd.DataFrame(ling_features_list)
ling_matrix = ling_df.values

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['text']).toarray()

X = np.hstack([ling_matrix, tfidf_matrix])
y = df['label'].values
feature_names = list(ling_df.columns) + [f"tfidf_{n}" for n in tfidf.get_feature_names_out()]

# 3. Train Ensemble
print("Training models...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(n_estimators=100, random_state=42)
ensemble = VotingClassifier([('rf', rf), ('xgb', xgb)], voting='soft')
ensemble.fit(X, y)

# 4. Save All Artifacts
model_dir = '../models/classical_model'
os.makedirs(model_dir, exist_ok=True)
joblib.dump(ensemble, os.path.join(model_dir, 'classical_model.pkl'))
joblib.dump(tfidf, os.path.join(model_dir, 'vectorizer.pkl'))
joblib.dump(feature_names, os.path.join(model_dir, 'feature_names.pkl'))

print(f"Success! Models and artifacts saved to {model_dir}")